In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch

import numpy as np
import h5py
import os
import glob
import re
from tqdm import tqdm
import pickle
from omegaconf import OmegaConf

import matplotlib.pyplot as plt

from lightning.pytorch import seed_everything

from snpgen.utils import instantiate_from_config
from snpgen.models.modules.utils import get_proper_state_dict_ddpm


OmegaConf.register_new_resolver("eval", eval)

plt.rcParams['figure.dpi'] = 200 # increase show resoultion

## User Settings

In [ ]:
# ============================================================
# USER SETTINGS - Configure these before running the notebook
# ============================================================

# Path to the DDPM checkpoint to load
reload_path = '/path/to/snpgen/checkpoints/trait1/trait1_ddpm_emb128_small_white-31744535/epoch=273-step=88228-loss=0.231.ckpt'

# Choose generation mode(s): 'complete', 'syn_recon', or 'augmented'
syn_dataset_type = ['complete', 'augmented']

# Sampling batch size for generation
batch_size = 2048 * 3

In [ ]:
seed = 42
seed_everything(seed, workers=True)

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu") 

In [ ]:
NUM_WORKERS = int(os.environ["SLURM_CPUS_PER_TASK"])
NUM_NODES = int(os.environ["SLURM_NNODES"])
ALLOCATED_GPUS_PER_NODE = int(os.environ["SLURM_GPUS_ON_NODE"])
SLURM_JOBID = os.environ["SLURM_JOB_ID"]

In [ ]:
num_gpus = torch.cuda.device_count()
print(f"{num_gpus} GPU(s) available")
print(f"Using {NUM_WORKERS} workers for the DataLoader")

# Load Config

In [ ]:
config_path = os.path.join(os.path.dirname(reload_path), 'config.yaml')

if os.path.exists(config_path):
    # Try to load config directly from the checkpoint directory
    print(f"Loading config from checkpoint directory: {config_path}")
    config = OmegaConf.load(config_path)
else:
    raise FileNotFoundError(
        f"config.yaml not found in checkpoint directory: {os.path.dirname(reload_path)}\n"
        "Please ensure the checkpoint directory contains a config.yaml file."
    )

In [ ]:
resolved_config_dict = OmegaConf.to_container(config, resolve=True)
config_orig = config.copy() # keep a backup of the original config prior to any change

In [ ]:
encoder_config = OmegaConf.to_container(config.model.params.first_stage_config.params.encoder_config.params, resolve=True)
decoder_config = OmegaConf.to_container(config.model.params.first_stage_config.params.decoder_config.params, resolve=True)

# Build Dataset

In [ ]:
if 'dataset_path' in config:
    h5_path = config['dataset_path']
    print(f"Using dataset path from reloaded config: {h5_path}")
    proj_name = os.path.basename(os.path.dirname(config['dataset_path'])).replace('ukb_', '')
    print(f"Inferred project name: {proj_name}")
else:
    raise ValueError("dataset_path not found in config. Please ensure the config.yaml contains a dataset_path entry with the appropriate path to the dataset.")

In [ ]:
print(f"Loading Dataset from: {h5_path}")
if config.get('data', {}).get('raw_dataset', None):
    if config.seed != seed:
        print(f"Overriding raw_dataset seed from {seed} to {config.seed}")
    raw_dataset = instantiate_from_config(config.data.raw_dataset, file_path=h5_path, seed=config.seed)
else:
    raise ValueError("raw_dataset config not found. Please ensure the config.yaml contains a data.raw_dataset section with the appropriate dataset configuration.")

In [ ]:
if config.get('data', {}).get('dataset', None):
    train_dataset = instantiate_from_config(config.data.dataset, raw_dataset.get_split('train'), block_ids=raw_dataset.get_metadata('full', 'block_id') if hasattr(raw_dataset, 'get_metadata') else None)
    val_dataset = instantiate_from_config(config.data.dataset, raw_dataset.get_split('val'), block_ids=raw_dataset.get_metadata('full', 'block_id') if hasattr(raw_dataset, 'get_metadata') else None)
    test_dataset = instantiate_from_config(config.data.dataset, raw_dataset.get_split('test'), block_ids=raw_dataset.get_metadata('full', 'block_id') if hasattr(raw_dataset, 'get_metadata') else None)
    
    complete_dataset = instantiate_from_config(config.data.dataset, raw_dataset.get_split('full'), block_ids=raw_dataset.get_metadata('full', 'block_id') if hasattr(raw_dataset, 'get_metadata') else None)

else:
    raise ValueError("dataset config not found. Please ensure the config.yaml contains a data.dataset section with the appropriate dataset configuration.")

# Build Models

In [ ]:
ddpm_training_wrapper = instantiate_from_config(config.model)

In [ ]:
ddpm_use_ema = True # whether to use the EMA weights at inference
ddpm_ema_dict = get_proper_state_dict_ddpm(reload_path, ema=ddpm_use_ema)
ddpm_training_wrapper.model.load_state_dict(ddpm_ema_dict, strict=True)

# Setup DataLoaders

In [ ]:
train_dataloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=True,
    #sampler=ImbalancedDatasetSampler(train_dataset, strategy='inverse_freq'), # balance dataset on labels (which also implicitly performs shuffling)
)

val_dataloader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=True,
)

test_dataloader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=True,
)

complete_dataloader = torch.utils.data.DataLoader(
    complete_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=True,
)

# Generate samples, compute reconstructions (only once!)
Needed only once, then we can save them for future analysis

In [ ]:
from snpgen.inference import (
    SyntheticDatasetGenerator,
    get_sampler,
    save_synthetic_dataset,
    get_output_filename,
    dataset_exists,
)

if not isinstance(syn_dataset_type, list):
    syn_dataset_type = [syn_dataset_type]
    
assert all(mode in ['complete', 'syn_recon', 'augmented'] for mode in syn_dataset_type), "Invalid choice in syn_dataset_type"

# ===================================================================================
# SETUP GENERATOR
# ===================================================================================

# Create generator
generator = SyntheticDatasetGenerator(
    model=ddpm_training_wrapper,
    config=config,
    decoder_config=decoder_config,
    device='cuda'
)
generator.prepare_model()

# ===================================================================================
# GENERATE DATASET
# ===================================================================================

for mode in syn_dataset_type:
    
    print(f"\n=== Generating dataset of type: {mode} ===")

    output_dir = os.path.dirname(reload_path)
    output_filename = get_output_filename(
        base_name='syn',
        mode=mode,
    )
    output_path = os.path.join(output_dir, output_filename)

    if not dataset_exists(output_dir, output_filename):
        
        if mode == 'complete':
            # Generate with same labels as original dataset
            print(f"Generating complete dataset...")
            result = generator.generate_complete(
                complete_dataloader,
            )
            
            save_synthetic_dataset(
                result=result,
                output_path=output_path,
                mode='complete',
            )

        elif mode == 'syn_recon':
            # Generate with reconstructions and latent space info
            print(f"Generating syn_recon dataset...")
            onehot = config.data.raw_dataset.params.get('onehot', True)
            result = generator.generate_syn_recon(
                val_dataloader,
                onehot=onehot,
            )
            
            save_synthetic_dataset(
                result=result,
                output_path=output_path,
                mode='syn_recon',
            )

        elif mode == 'augmented':
            # Generate with augmented label distribution (binary balanced)
            print(f"Generating augmented dataset...")
            
            # Get original labels for reference
            original_labels = complete_dataset.targets
            
            # Binary: generate balanced dataset with 2 * num_controls samples
            num_controls = (original_labels == 0).sum()
            n_samples = 2 * num_controls
            
            print(f"Original dataset size: {len(original_labels)}")
            print(f"Augmented dataset size: {n_samples}")
            
            sampler = get_sampler(
                strategy='binary_balanced',
                original_labels=original_labels,
                n_classes=2
            )
            
            # Generate samples
            result = generator.generate_augmented(
                label_sampler=sampler,
                total_samples=n_samples,
                batch_size=batch_size,
                seq_len=config.seq_len,
            )
            
            save_synthetic_dataset(
                result=result,
                output_path=output_path,
                mode='augmented',
                augmentation_strategy=sampler.name
            )

        print(f"\nSaved dataset to: {output_path}")
        print(f"Generated {len(result.samples)} samples")
        
    else:
        print(f"Dataset already exists: {output_path}")